# Fabula — Batched Teacher Model Expansion (Tasks C-02 & C-03)

This notebook takes the 300 deterministic symbolic mystery skeletons (`clean_skeletons.jsonl`) and executes batched semantic enrichment using `Qwen/Qwen2.5-7B-Instruct` (Apache 2.0) on Kaggle Dual Tesla T4 GPUs.

### Pipeline Overview:
1. **Load `Qwen/Qwen2.5-7B-Instruct`** in Float16 across Dual T4 GPUs (`device_map="auto"`).
2. **Batched Narrative Enrichment:** Generates atmospheric location descriptions, NPC background depth, and clue details (Batch size = 8).
3. **Decompose into 4 SFT Stages:** Converts each scenario into the 4 training stages (`world`, `cast`, `truth+clues`, `identity+memory`).
4. **Disjoint Seed Partitioning:** Exports `train.jsonl` (2,048), `val.jsonl` (256), and `test.jsonl` (254).
5. **Runtime:** ~18.3 minutes on Kaggle Dual T4s.

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers accelerate bitsandbytes
print("Dependencies installed successfully!")

In [ ]:
# Step 2: Load Model & Tokenizer on Dual Tesla T4 GPUs
import os
import time
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {MODEL_ID} across available GPUs...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"Model loaded in {round(time.time() - t0, 1)}s!")

In [ ]:
# Step 3: Run Batched Teacher Expansion & Export SFT Splits
# Execute the teacher expansion script
!python training/scripts/kaggle_teacher_expansion.py